# 02. 텐서 데이터셋 준비

**Phase 1 — Step 2: log-return → LSTM 학습용 PyTorch 텐서 변환**

## 이 노트북의 역할

1. 분석 기간(2016-01-01 ~ 2025-12-31) log-return을 CSV에서 로드한다
2. `scripts/dataset.py` 유틸리티 함수를 검증한다
   - `make_sequences`: 슬라이딩 윈도우 (X, y) 생성
   - `walk_forward_folds`: Walk-Forward 폴드 인덱스 생성
   - `build_fold_datasets`: 폴드별 스케일링 + `LSTMDataset` 생성
3. DataLoader 배치 반복이 정상 동작하는지 확인한다
4. 전체 106개 폴드 일괄 생성 후 형태 이상 없음을 검증한다

## 누수 방지 설계

- `StandardScaler.fit()` 은 **훈련 구간(train_idx)** 데이터로만 수행
- 테스트 시퀀스 입력이 purge/embargo 기간과 겹쳐도 **레이블이 아닌 입력 피처** 이므로 허용
- Walk-Forward 폴드 인덱스는 0-based 정수이므로 날짜와 무관하게 재사용 가능

## 진실원

유틸리티 구현은 [scripts/dataset.py](scripts/dataset.py) 에 있다. 이 노트북은 **시연 및 검증** 목적이다.

## §1. 환경 설정 로드

In [1]:
%run ./00_setup_and_utils.ipynb

[OK] scripts.setup import 완료 — BASE_DIR = /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_LSTM
[OK] 한글 폰트 설정 완료: AppleGothic
[OK] 시드 고정 완료: SEED=42
[정보] PyTorch 버전: 2.11.0, CUDA 가용: False
[OK] 경로 상수 확인
  BASE_DIR      = /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_LSTM
  RAW_DATA_DIR  = /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_LSTM/results/raw_data
  SETTING_A_DIR = /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_LSTM/results/setting_A
  SETTING_B_DIR = /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_LSTM/results/setting_B
[OK] 공통 import + 표시 옵션 적용 완료
  pandas 2.3.3, numpy 2.4.4
  Phase 1 — LSTM 단독 베이스라인 / 환경 설정 완료
  한글 폰트  : AppleGothic
  시드       : 42
  결과 경로  : /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_LSTM/results
  진실원     : scripts/se

## §2. 분석 데이터 로드

log_return 은 **전체 기간** 데이터에서 먼저 계산한 뒤 분석 기간으로 필터링한다.
이렇게 해야 분석 시작일(2016-01-04)의 log_return 이 직전 거래일(2015-12-31) 대비로
올바르게 계산된다. 분석 기간 내 NaN 이 0개임을 §3에서 확인한다.

In [2]:
ANALYSIS_START = '2016-01-01'
ANALYSIS_END   = '2025-12-31'

raw_dict = {}
for tic in ['SPY', 'QQQ']:
    df = pd.read_csv(RAW_DATA_DIR / f'{tic}.csv', index_col=0, parse_dates=True)
    df['log_return'] = np.log(df['Adj Close']).diff()  # 전체 기간에서 계산 → 분석 시작일 NaN 방지
    raw_dict[tic] = df.loc[ANALYSIS_START:ANALYSIS_END]

print(f'{"티커":>6}  {"행수":>6}  {"NaN":>4}  {"시작일":>12}  {"종료일":>12}')
print('-' * 55)
for tic, df in raw_dict.items():
    lr = df['log_return']
    print(f'{tic:>6}  {len(df):>6}  {lr.isna().sum():>4}  '
          f'{str(df.index[0].date()):>12}  {str(df.index[-1].date()):>12}')

    티커      행수   NaN           시작일           종료일
-------------------------------------------------------
   SPY    2514     0    2016-01-04    2025-12-31
   QQQ    2514     0    2016-01-04    2025-12-31


## §3. `scripts.dataset` import

`scripts/dataset.py` 의 공개 인터페이스를 로드하고 각 함수의 역할을 확인한다.

| 함수 / 클래스 | 역할 |
|---|---|
| `LSTMDataset` | `(X, y)` 텐서를 보관하는 `torch.utils.data.Dataset` |
| `make_sequences` | 1D/2D 시계열 → 슬라이딩 윈도우 `(X, y)` |
| `walk_forward_folds` | Walk-Forward 폴드 인덱스 목록 생성 |
| `build_fold_datasets` | 폴드 하나의 스케일링 + `LSTMDataset` 일괄 생성 |

In [3]:
from scripts.dataset import (
    LSTMDataset,
    make_sequences,
    walk_forward_folds,
    build_fold_datasets,
)
from torch.utils.data import DataLoader

print('[OK] scripts.dataset import 완료')

[OK] scripts.dataset import 완료


## §4. `make_sequences` — 슬라이딩 윈도우 시연

Setting A 의 `seq_len = 21` (약 1개월 lookback)을 사용한다.

각 샘플 구조:
```
X[i] = log_return[i : i+21]     # 과거 21일 수익률 시퀀스  shape: (21, 1)
y[i] = log_return[i+21]          # 다음 날(i+21) 수익률       scalar
```

샘플 수  N = T − seq_len − horizon + 1 = 2514 − 21 − 1 + 1 = **2493**

In [4]:
SEQ_LEN = 21

spy_lr = raw_dict['SPY']['log_return'].values   # shape (2514,), NaN 없음
qqq_lr = raw_dict['QQQ']['log_return'].values

X_spy, y_spy = make_sequences(spy_lr, seq_len=SEQ_LEN)

print('make_sequences(spy_lr, seq_len=21) 결과')
print(f'  X.shape = {X_spy.shape}   → (N, seq_len, n_features)')
print(f'  y.shape = {y_spy.shape}')
print()
print(f'X[0] 첫 5개 값: {X_spy[0].flatten()[:5].round(6)}  ...  (총 {SEQ_LEN}개)')
print(f'y[0]  = {y_spy[0]:.6f}  (X[0] 다음 날 log_return)')
print()
print(f'검증: X[0][-1] == spy_lr[20] → {X_spy[0][-1, 0]:.6f} == {spy_lr[20]:.6f}')
print(f'검증: y[0]     == spy_lr[21] → {y_spy[0]:.6f} == {spy_lr[21]:.6f}')

make_sequences(spy_lr, seq_len=21) 결과
  X.shape = (2493, 21, 1)   → (N, seq_len, n_features)
  y.shape = (2493,)

X[0] 첫 5개 값: [-0.014078  0.00169  -0.012695 -0.024284 -0.011037]  ...  (총 21개)
y[0]  = 0.005977  (X[0] 다음 날 log_return)

검증: X[0][-1] == spy_lr[20] → -0.018186 == -0.018186
검증: y[0]     == spy_lr[21] → 0.005977 == 0.005977


## §5. `walk_forward_folds` — Walk-Forward 폴드 생성

Setting A 파라미터를 그대로 사용한다.

| 파라미터 | 값  | 의미 |
|---------|-----|------|
| IS      | 231 | 훈련 구간 (~11개월) |
| purge   | 21  | 레이블 겹침 제거 |
| emb     | 21  | 추가 엠바고 |
| OOS     | 21  | 테스트 구간 (~1개월) |
| step    | 21  | 폴드 이동 간격 |

예상 폴드 수 = ⌊(2514 − 294) / 21⌋ + 1 = **106**

In [5]:
IS, PURGE, EMB, OOS, STEP = 231, 21, 21, 21, 21
n_samples = len(spy_lr)

folds = walk_forward_folds(n_samples, IS, PURGE, EMB, OOS, STEP)
print(f'총 폴드 수: {len(folds)}')
print()

# 날짜 인덱스 (시각적 확인용)
dates = raw_dict['SPY'].index

print(f'{"폴드":>4}  {"train 시작":>12}  {"train 끝":>12}  {"test 시작":>12}  {"test 끝":>12}')
print('-' * 65)
for fold_i in [0, 1, 2, len(folds) - 1]:
    tr, te = folds[fold_i]
    prefix = '...' if fold_i == len(folds) - 1 else ''
    print(f'{fold_i:>4}  {str(dates[tr[0]].date()):>12}  {str(dates[tr[-1]].date()):>12}  '
          f'{str(dates[te[0]].date()):>12}  {str(dates[te[-1]].date()):>12}')

print()
tr0, te0 = folds[0]
print(f'[폴드 0 상세]')
print(f'  train: 인덱스 {tr0[0]}~{tr0[-1]}  ({len(tr0)}일)')
print(f'  test : 인덱스 {te0[0]}~{te0[-1]}  ({len(te0)}일)')
print(f'  간격 (purge+emb): {te0[0] - tr0[-1] - 1}일')

총 폴드 수: 106

  폴드      train 시작       train 끝       test 시작        test 끝
-----------------------------------------------------------------
   0    2016-01-04    2016-11-30    2017-02-02    2017-03-03
   1    2016-02-03    2016-12-30    2017-03-06    2017-04-03
   2    2016-03-04    2017-02-01    2017-04-04    2017-05-03
 105    2024-10-08    2025-09-10    2025-11-10    2025-12-09

[폴드 0 상세]
  train: 인덱스 0~230  (231일)
  test : 인덱스 273~293  (21일)
  간격 (purge+emb): 42일


## §6. `build_fold_datasets` — 폴드별 텐서 생성

첫 번째 폴드를 예시로 `LSTMDataset` 생성 과정을 검증한다.

**훈련 시퀀스** (`make_sequences` 경유):
```
N_train = IS − seq_len − 1 + 1 = 231 − 21 = 210
X_train.shape = (210, 21, 1)
```

**테스트 시퀀스** (t ∈ test_idx 각각에 대해 독립 구성):
```
X[j] = scaled[t_j − 21 : t_j]   입력 구간이 embargo와 겹쳐도 허용 (입력 피처, 레이블 아님)
y[j] = scaled[t_j, 0]            당일 log_return (예측 타깃)
N_test = OOS = 21
```

In [6]:
train_idx, test_idx = folds[0]

train_ds, test_ds, scaler = build_fold_datasets(
    spy_lr, train_idx, test_idx, SEQ_LEN
)

print('=== 폴드 0 텐서 형태 ===')
print(f'train_ds.X : {tuple(train_ds.X.shape)}   (N_train, seq_len, n_feat)')
print(f'train_ds.y : {tuple(train_ds.y.shape)}')
print(f'test_ds.X  : {tuple(test_ds.X.shape)}    (N_test, seq_len, n_feat)')
print(f'test_ds.y  : {tuple(test_ds.y.shape)}')
print()
print('[스케일러 — 훈련 구간 기준]')
print(f'  mean  = {scaler.mean_[0]:.8f}   (log_return 평균)')
print(f'  scale = {scaler.scale_[0]:.8f}  (log_return 표준편차)')
print()

# dtype 확인
print('[텐서 dtype]')
print(f'  X dtype: {train_ds.X.dtype}  (float32 필요)')
print(f'  y dtype: {train_ds.y.dtype}')

=== 폴드 0 텐서 형태 ===
train_ds.X : (210, 21, 1)   (N_train, seq_len, n_feat)
train_ds.y : (210,)
test_ds.X  : (21, 21, 1)    (N_test, seq_len, n_feat)
test_ds.y  : (21,)

[스케일러 — 훈련 구간 기준]
  mean  = 0.00040364   (log_return 평균)
  scale = 0.00845726  (log_return 표준편차)

[텐서 dtype]
  X dtype: torch.float32  (float32 필요)
  y dtype: torch.float32


## §7. DataLoader 배치 검증

`torch.utils.data.DataLoader` 로 배치 반복이 정상 동작하는지 확인한다.

- `shuffle=True` → 훈련 시 시간 순서 무작위화 (LSTM 학습 관행)
- `shuffle=False` → 테스트 시 시간 순서 보존

In [7]:
BATCH_SIZE = 32

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=len(test_ds), shuffle=False)

print(f'train_loader: {len(train_loader)} 배치 (batch_size={BATCH_SIZE}, N={len(train_ds)})')
print(f'test_loader : {len(test_loader)} 배치  (전체 한 번에)')
print()

for xb, yb in train_loader:
    print(f'[train 첫 배치] X: {tuple(xb.shape)},  y: {tuple(yb.shape)},  dtype: {xb.dtype}')
    break

for xb, yb in test_loader:
    print(f'[test  전체  ] X: {tuple(xb.shape)},  y: {tuple(yb.shape)}')

train_loader: 7 배치 (batch_size=32, N=210)
test_loader : 1 배치  (전체 한 번에)

[train 첫 배치] X: (32, 21, 1),  y: (32,),  dtype: torch.float32
[test  전체  ] X: (21, 21, 1),  y: (21,)


## §8. 전체 106 폴드 일괄 검증

모든 폴드에 대해 `build_fold_datasets` 를 실행하고 형태 이상이 없는지 확인한다.
실제 학습은 `03_setting_A_daily21.ipynb` 에서 수행한다.

In [8]:
n_train_list, n_test_list = [], []

for tr_idx, te_idx in folds:
    tr_ds, te_ds, _ = build_fold_datasets(spy_lr, tr_idx, te_idx, SEQ_LEN)
    n_train_list.append(len(tr_ds))
    n_test_list.append(len(te_ds))

print(f'총 {len(folds)} 폴드 정상 생성 완료')
print(f'train 샘플 수: 고정={len(set(n_train_list)) == 1},  값={n_train_list[0]}')
print(f'test  샘플 수: 고정={len(set(n_test_list)) == 1},   값={n_test_list[0]}')

총 106 폴드 정상 생성 완료
train 샘플 수: 고정=True,  값=210
test  샘플 수: 고정=True,   값=21


## §9. 멀티변량 시연 — SPY + QQQ 결합

`extra_features` 파라미터를 사용해 두 티커 수익률을 합산한 경우를 보인다.

- `n_features = 2` (SPY log_return + QQQ log_return)
- 스케일러는 여전히 훈련 구간으로만 fit

In [9]:
train_idx_0, test_idx_0 = folds[0]

tr_mv, te_mv, scaler_mv = build_fold_datasets(
    spy_lr, train_idx_0, test_idx_0, SEQ_LEN,
    extra_features=qqq_lr[:, np.newaxis],  # QQQ 를 추가 피처로
)

print('[멀티변량: SPY + QQQ]')
print(f'train_ds.X : {tuple(tr_mv.X.shape)}   → n_features=2')
print(f'test_ds.X  : {tuple(te_mv.X.shape)}')
print(f'scaler mean : {scaler_mv.mean_.round(8)}  (SPY, QQQ 각각)')

[멀티변량: SPY + QQQ]
train_ds.X : (210, 21, 2)   → n_features=2
test_ds.X  : (21, 21, 2)
scaler mean : [0.00040364 0.00024811]  (SPY, QQQ 각각)


## §10. 결론 · 체크포인트

### 생성된 텐서 구조 (단변량 Setting A)

| 구분 | shape | 설명 |
|------|-------|------|
| `X_train` | `(210, 21, 1)` | 210개 훈련 시퀀스, 각 21일, 단변량 |
| `y_train` | `(210,)` | 다음 날 scaled log_return |
| `X_test`  | `(21, 21, 1)` | OOS 21일 각각에 대한 예측 시퀀스 |
| `y_test`  | `(21,)` | OOS 당일 scaled log_return |

### 다음 노트북에 전달할 사실

1. `scripts.dataset` 세 함수 정상 동작 확인 완료
2. 폴드 수: **106** (SPY·QQQ 동일)
3. 스케일러: 폴드별 별도 생성 — 역변환 시 `scaler.inverse_transform(y_pred.reshape(-1,1))[:, 0]`
4. 멀티변량: `build_fold_datasets(..., extra_features=qqq_lr[:, np.newaxis])` 형태로 확장 가능

### WORKLOG 업데이트 필요 항목

- Step 2 실행 결과: 텐서 형태, 폴드 수, 정상 생성 여부 기록